In [1]:
import duckdb
import os
from pathlib import Path

# ensuring that os.chdir is idempotent and that we are in the project root directory,
# not inside the notebooks directory
if 'notebooks' not in os.listdir(Path.cwd()):
    print("Still inside notebooks directory, changing to project root directory.")
    os.chdir(Path.cwd().parent)
    print("Current working directory after change: ", Path.cwd())
else:
    print(f"Already in parent directory (current working directory: {Path.cwd()})")


# load config
from src.io.load_config import load_config
sw_config = load_config()['space_weather']

# load T1 and T2 dataset dir
T1_dir = sw_config['preprocessing']['k_index']['T1_output_dir']
T2_dir = sw_config['transform']['k_index']['T2_output_dir']

Still inside notebooks directory, changing to project root directory.
Current working directory after change:  d:\data-sci-projects\space-weather-project-scrub


# Test endpoints from CDAWeb HAPI

CDAWeb HAPI description can be found at https://cdaweb.gsfc.nasa.gov/hapi

In [2]:
import pandas as pd
import requests
from io import StringIO

## `/info`

In [3]:
def fetch_info_omni_hro2_1min(dataset_id="OMNI_HRO2_1MIN",
                            base_url="https://cdaweb.gsfc.nasa.gov/hapi",
                            ):

    # as per HAPI description, endpoints respond to HTTP GET requests 
    response = requests.get(base_url + '/info',
                            params={"id": dataset_id},
                            timeout=120)
    
    response.raise_for_status()

    return response.json()

In [4]:
info_endpoint_test = fetch_info_omni_hro2_1min()

In [9]:
info_endpoint_test

{'HAPI': '2.0',
 'resourceURL': 'https://cdaweb.gsfc.nasa.gov/misc/NotesO.html#OMNI_HRO2_1MIN',
 'contact': 'J.H. King, N. Papatashvilli @ AdnetSystems, NASA GSFC',
 'parameters': [{'name': 'Time',
   'length': 24,
   'units': 'UTC',
   'type': 'isotime',
   'fill': None},
  {'name': 'IMF',
   'description': 'OMNI ID code for the source spacecraft for time-shifted IMF values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'PLS',
   'description': 'OMNI ID code for the source spacecraft  for time-shifted IP plasma values (see OMNI documentation link for codes)',
   'units': None,
   'type': 'integer',
   'fill': '99'},
  {'name': 'IMF_PTS',
   'description': 'Number of fine time scale points in IMF averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'name': 'PLS_PTS',
   'description': 'Number of fine time scale points in plasma averages',
   'units': None,
   'type': 'integer',
   'fill': '999'},
  {'na

In [10]:
pd.DataFrame(info_endpoint_test['parameters'])

,name,length,units,type,fill,description
0,Time,24.0,UTC,isotime,None,NaN
1,IMF,NaN,None,integer,99,OMNI ID code for the source spacecraft for tim...
2,PLS,NaN,None,integer,99,OMNI ID code for the source spacecraft for ti...
3,IMF_PTS,NaN,None,integer,999,Number of fine time scale points in IMF averages
4,PLS_PTS,NaN,None,integer,999,Number of fine time scale points in plasma ave...
5,percent_interp,NaN,None,integer,999,Percent interpolated
6,Timeshift,NaN,seconds,integer,999999,Timeshift (seconds)
7,RMS_Timeshift,NaN,seconds,integer,999999,RMS Timeshift (seconds)
8,RMS_phase,NaN,nT,double,99.99,"RMS, Phase front normal (nT)"
9,Time_btwn_obs,NaN,seconds,integer,999999,Time between observations (seconds)


In [ ]:
query_var = 'BX_GSE'
display(
    pd.DataFrame(info_endpoint_test['parameters']).query(f"name=='{query_var}'")
)

display(
    pd.DataFrame(info_endpoint_test['parameters']).query(f"name=='{query_var}'").fill.item()
)


## `/data` (with params)

In [6]:
# https://cdaweb.gsfc.nasa.gov/hapi/data?id=OMNI_HRO2_1MIN&parameters=F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure&time.min=2021-11-21T00:00:00Z&time.max=2021-11-22T00:00:00Z&format=csv
def fetch_data_omni_hro2_1min(start_utc: str,
                         end_utc: str,
                         vars: str="F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure"
                         ):

    url = "https://cdaweb.gsfc.nasa.gov/hapi/data"

    params = {
        "id": "OMNI_HRO2_1MIN",
        "time.min": start_utc,
        "time.max": end_utc
    }
    
    if vars:
        # no intermediate or trailing whitespaces allowed
        vars = vars.replace(" ", "")
        params['parameters'] = vars

    print("Requesting solar dataset from OMNI_HRO2_1MIN...")
    response = requests.get(url, params=params, timeout=120)
    response.raise_for_status()
    print("Request succeeded.")

    df = pd.read_csv(StringIO(response.text),
                     comment="#",
                     header=None)
    df.columns = ["time"] + vars.split(',')
    return df


In [ ]:
from datetime import datetime, timezone, timedelta

In [8]:
omni = fetch_data_omni_hro2_1min(
    start_utc="2021-11-21T00:00:00Z",
    end_utc="2021-11-22T00:00:00Z"
)
omni

Requesting solar dataset from OMNI_HRO2_1MIN...
Request succeeded.


,time,F,BX_GSE,BY_GSM,BZ_GSM,flow_speed,proton_density,Pressure
0,2021-11-21T00:00:00.000Z,4.79,1.87,-2.40,-3.37,99999.9,999.99,99.99
1,2021-11-21T00:01:00.000Z,5.73,4.25,0.29,2.60,99999.9,999.99,99.99
2,2021-11-21T00:02:00.000Z,4.90,1.72,-3.04,-3.39,583.6,4.70,3.20
3,2021-11-21T00:03:00.000Z,4.92,2.03,-3.04,-3.23,583.6,4.70,3.20
4,2021-11-21T00:04:00.000Z,5.00,1.68,-2.93,-3.67,583.6,4.70,3.20
...,...,...,...,...,...,...,...,...
1435,2021-11-21T23:55:00.000Z,3.69,2.28,-2.76,0.48,612.4,2.33,1.75
1436,2021-11-21T23:56:00.000Z,3.76,1.76,-3.15,0.31,619.3,2.45,1.88
1437,2021-11-21T23:57:00.000Z,3.75,-0.65,-3.07,-1.14,635.0,2.35,1.90
1438,2021-11-21T23:58:00.000Z,3.95,-1.41,-3.01,-2.11,643.1,2.31,1.91
